In [2]:
!unzip -q entropy-engine.zip

In [3]:
# 1. Enter the directory
%cd /content/entropy-engine

/content/entropy-engine


In [4]:
!ls -l

total 20
drwxr-xr-x 3 root root 4096 Mar 28 21:07 csrc
drwxr-xr-x 2 root root 4096 Mar 28 21:07 python
-rw-r--r-- 1 root root  443 Mar 28 21:07 setup.py
-rw-r--r-- 1 root root 1279 Mar 28 21:07 test_cubin.py
-rw-r--r-- 1 root root 1525 Mar 28 21:07 test_engine.py


In [4]:
# 2. Install build dependencies
!pip install ninja cmake transformers datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 16.6 MB/s eta 0:00:00


In [5]:
!pip install torch triton transformers

In [6]:
!pip install pybind11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 26.9 MB/s eta 0:00:00


In [51]:

# 3. Generate the bare-metal Triton binaries (.cubin and .h)
!python python/compile_kernels.py

🚀 Triggering JIT-to-AOT Bypass...

Compiling gemm_int8...
  [CRITICAL] shared_mem = 36864 bytes

Compiling gemm_fp8...
  [CRITICAL] shared_mem = 49152 bytes

Compiling gemm_fp16...
  [CRITICAL] shared_mem = 49152 bytes

✅ AOT Dump Complete! Ready for C++.


In [55]:
%cd /content/entropy-engine

# 1. Recompile the C++ Engine (This overrides the old one)
!pip install -e .

/content/entropy-engine
Obtaining file:///content/entropy-engine
  Preparing metadata (setup.py) ... done
  Running setup.py develop for entropy_engine_cuda


In [56]:
# 2. Run the bare-metal benchmark!
!python test_cubin.py

Reading AOT metadata from /content/entropy-engine/csrc/generated/gemm_int8_meta.json
  Metadata: {
  "shared": 36864,
  "num_warps": 8,
  "name": "gemm_int8_kernel",
  "BLOCK_M": 64,
  "BLOCK_N": 128,
  "BLOCK_K": 64
}
  Using: shared_mem=36864, num_warps=8

Setting up tensors: M=1024, K=4096, N=4096
  a_int8:  torch.Size([1024, 4096]) torch.int8 stride=(4096, 1)
  b_int8:  torch.Size([4096, 4096]) torch.int8 stride=(4096, 1)
  a_scale: torch.Size([1024]) torch.float32
  b_scale: torch.Size([4096]) torch.float32

Initializing CubinEngine...
[from_metadata] shared=36864 warps=8 BLOCK_M=64 BLOCK_N=128
[CubinEngine] Loaded: /content/entropy-engine/csrc/generated/gemm_int8.cubin
  func=gemm_int8_kernel shared=36864 warps=8 BLOCK_M=64 BLOCK_N=128
  sizeof(TritonABIArgs)=80
Smoke test...
  Output dtype: torch.float16 (CORRECT)
  Output shape: torch.Size([1024, 4096])
  Max error vs reference: 0.2500
  Mean error vs reference: 0.000018
  CORRECTNESS: PASS

Benchmarking cuBLAS FP16 baseline...

In [58]:
%cd /content/entropy-engine/python


/content/entropy-engine/python


In [62]:
!python benchmark.py


╔══════════════════════════════════════════════════════════╗
║    Entropy-Gated Dispatch v2 — L4 / Llama-3 / 3-Tier   ║
╚══════════════════════════════════════════════════════════╝

  Device: NVIDIA L4
  Compute capability: sm_89
  CUDA: 12.8

─── CORRECTNESS TESTS ──────────────────────────────────────

TEST: FP16 SRAM Upcast GEMM
  Shape: [256,4096] × [4096,4096]
  Max abs error:  0.250000
  Mean abs error: 0.013928
  Status: PASS

TEST: Native INT8 GEMM
  Shape: [256,4096] × [4096,4096]
  Max abs error (vs weight-quant ref): 5.750000
  Mean abs error: 0.874506
  Status: PASS

TEST: FP8 e4m3 GEMM
  Shape: [256,4096] × [4096,4096]
  Max abs error (vs weight-quant ref): 0.129395
  Mean abs error: 0.019610
  Status: PASS

TEST: Fused Softmax + Entropy
  Shape: logits[4,32,512,512]
  Max prob error:    0.00006104
  Max entropy error: 0.000000
  Ref entropy:   [4.762599945068359, 4.761904239654541]...
  Fused entropy: [4.762599945068359, 4.761904239654541]...
  Status: PASS

TEST: Pipeli

In [17]:
!python quality_eval.py --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --max-samples 20


╔══════════════════════════════════════════════════════════╗
║   Phase 2: 3-Tier Quality Evaluation (Llama + WikiText) ║
╚══════════════════════════════════════════════════════════╝

  Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:00<00:00, 549.78it/s, Materializing param=model.norm.weight]
  Parameters: 1100.0M
  hidden_size=2048, intermediate=5632
  layers=22, heads=32
  KV heads=4 (GQA)
  Loading WikiText-2 test set...
  Tokenizing...
Token indices sequence length is longer than the specified maximum sequence length for this model (338535 > 2048). Running this sequence through the model will result in indexing errors
  Total tokens: 338,535
  Chunks: 20 × ~1024 tokens

─── BASELINE: FP16 (original model) ────────────────────────
  Perplexity: 8.51
  Time: 3.0s

─── ENTROPY PROFILE ────────────────────────────────────────
  Layer        Mean      Min      Max
  ──────── ──────── ──────── ────────
  0

In [63]:
!cd /content && zip -r /content/entropy-engine.zip entropy-engine/ -x "entropy-engine/build/*" "entropy-engine/.git/*" "entropy-engine/__pycache__/*"

from google.colab import files
files.download('/content/entropy-engine.zip')

updating: entropy-engine/ (stored 0%)
updating: entropy-engine/python/ (stored 0%)
updating: entropy-engine/python/benchmark.py (deflated 73%)
updating: entropy-engine/python/kernels.py (deflated 75%)
updating: entropy-engine/python/__pycache__/ (stored 0%)
updating: entropy-engine/python/__pycache__/kernels.cpython-312.pyc (deflated 66%)
updating: entropy-engine/python/compile_kernels.py (deflated 71%)
updating: entropy-engine/python/quality_eval.py (deflated 71%)
updating: entropy-engine/python/dispatch.py (deflated 72%)
updating: entropy-engine/setup.py (deflated 48%)
updating: entropy-engine/entropy_engine_cuda.egg-info/ (stored 0%)
updating: entropy-engine/entropy_engine_cuda.egg-info/dependency_links.txt (stored 0%)
updating: entropy-engine/entropy_engine_cuda.egg-info/PKG-INFO (deflated 11%)
updating: entropy-engine/entropy_engine_cuda.egg-info/SOURCES.txt (deflated 47%)
updating: entropy-engine/entropy_engine_cuda.egg-info/top_level.txt (stored 0%)
updating: entropy-engine/test

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>